# 🔎 Clase 04 · Práctica guiada: cómo identifico datos faltantes
Dataset: **clientes de un banco** (`clientes_banco.csv`, sintético). Es el análogo al dataset de *churn* de tarjetas de crédito que usó el profesor:
clientes que se van (`abandono = 1`) y clientes que se quedan.

Igual que en la clase, el dataset **no trae nulos**, así que los vamos a **simular** y luego haremos el recorrido completo:

1. Simular nulos (algunos en las **mismas filas**).
2. Reporte de nulos (tabla + gráficos).
3. Debate: ¿se pueden imputar todas las variables?
4. Identificar el tipo: **correlación de nulidad** (heatmap) y dendrograma.
5. Imputar: ffill/bfill, mediana (el "pico"), **KNNImputer** e **IterativeImputer**.
6. Eliminar duplicados.

Ejecuta celda por celda (`Shift + Enter`). En las celdas **✏️ Tu turno** escribes tú.
Al final implementa `ejercicio.py` y evalúa con
`python -m framework evaluar cursos/nivelacion/n1-python-data-science/m1-eda-faltantes/clase-04-identificar-faltantes`.

In [ ]:
import sys
from pathlib import Path

# Localiza la raíz del repo, para que el notebook funcione desde cualquier carpeta
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "framework").exists())
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from framework.testing import ruta_dataset  # si el CSV no existe, lo genera

df = pd.read_csv(ruta_dataset("clientes_banco.csv"))
df = df.drop(columns=["cliente_id"])   # como el profe: quitamos lo que no aporta (aquí, un identificador)
df.info()

`info()` dice **5000 entradas** y todas las columnas con 5000 *non-null*: no hay nulos.

### Antes de simular: el faltante "disfrazado"
En el dataset del profesor, `Marital_Status` no tenía NaN pero sí una categoría **Unknown / desconocido** (749 filas).
Eso **también es un dato faltante**: no sabemos si esos 749 son casados, solteros o divorciados, así que la variable
**ya tiene sesgo** antes de hacer nada. Por eso siempre revisa `value_counts()` de las categóricas:

In [ ]:
df["segmento"].value_counts()

Aquí `segmento` está limpio (Masivo / Preferente / Premium). Si apareciera algo como `"Desconocido"`, `"N/A"` o `"-"`,
sería un faltante disfrazado (cómo convertirlos a NaN lo practicas en la clase 03).

## Paso 1 · Simular nulos (como el profesor)
El profesor borró el **60 %** de `Dependent_count` y `Marital_Status` **en las mismas filas**, y otros pocos en
`Credit_Limit`, `Total_Trans_Ct` y `Total_Trans_Amt`. Nosotros hacemos lo mismo:

| Columna | Qué le hacemos | Tipo que "fabricamos" |
|---|---|---|
| `edad` y `num_productos` | 60 % de NaN en las **mismas filas** | patrón conjunto (MAR, como en la clase) |
| `ingreso_mensual` | 10 % de NaN al azar | MCAR |
| `reclamos_ultimo_anio` | 8 % de NaN al azar (otra semilla) | MCAR |
| `antiguedad_anios` | falta más en clientes que **abandonaron** | 📌 Extra: MAR respecto de una columna **completa** |

La clave para que sea reproducible es `random_state`: con la misma semilla salen siempre las mismas filas.

In [ ]:
dfn = df.copy()

filas = dfn.sample(frac=0.6, random_state=123).index          # 60 % de las filas
dfn.loc[filas, ["edad", "num_productos"]] = np.nan              # MISMAS filas en ambas columnas

dfn.loc[dfn.sample(frac=0.10, random_state=1).index, "ingreso_mensual"] = np.nan
dfn.loc[dfn.sample(frac=0.08, random_state=2).index, "reclamos_ultimo_anio"] = np.nan

rng = np.random.default_rng(7)
prob = np.where(dfn["abandono"] == 1, 0.5, 0.03)                # si abandonó, falta mucho más
dfn.loc[rng.random(len(dfn)) < prob, "antiguedad_anios"] = np.nan

dfn.info()

Ahora `edad` y `num_productos` tienen 2000 *non-null* (les quitamos 3000 = 60 %), igual que en la clase
`Dependent_count` quedó con 4051 de 10127.

## Paso 2 · El reporte de nulos
El profesor armó un reporte más ordenado que `info()`: **variable · cantidad de nulos · no nulos · porcentaje**
(porcentaje × 100 y redondeado a 3 decimales). Los ingredientes:

In [ ]:
nulos = dfn.isna().sum()                       # cuántos NaN por columna
porcentaje = (nulos / len(dfn) * 100).round(3) # en % con 3 decimales
pd.concat([nulos, porcentaje], axis=1, keys=["nulos", "porcentaje"])

➡️ En el **ejercicio 1** (`reporte_nulos`) lo conviertes en una función que devuelve las 4 columnas, ordenadas
de más a menos nulos.

### Gráfico de barras (lo que hace `missingno.bar`)

In [ ]:
pct = dfn.isna().mean().mul(100).sort_values()
ax = pct.plot.barh(figsize=(7, 4), color=["tab:red" if v > 50 else "tab:blue" for v in pct])
ax.axvline(50, color="gray", ls="--", label="50 %")
ax.set_xlabel("% de nulos"); ax.set_title("Porcentaje de nulos por variable"); ax.legend()
plt.tight_layout(); plt.show()

### Matriz de nulidad (lo que hace `missingno.matrix`)
Cada fila del dataset es una línea horizontal; **blanco = falta el dato**. Mira cómo `edad` y `num_productos`
tienen los huecos exactamente a la misma altura.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.imshow(dfn.isna().to_numpy(), aspect="auto", cmap="gray_r", interpolation="nearest")
ax.set_xticks(range(dfn.shape[1])); ax.set_xticklabels(dfn.columns, rotation=60, ha="right")
ax.set_ylabel("fila"); ax.set_title("Matriz de nulidad (negro = falta)")
plt.tight_layout(); plt.show()

### (Opcional) Los gráficos con `missingno`
El profesor usó la librería **missingno**. No viene en `requirements.txt`; si la quieres probar:
`pip install missingno`. Si no está instalada, la celda solo muestra un aviso.

In [ ]:
try:
    import missingno as msno
    msno.bar(dfn, figsize=(8, 3)); plt.show()
    msno.matrix(dfn, figsize=(8, 3)); plt.show()
    msno.heatmap(dfn, figsize=(5, 4)); plt.show()
    msno.dendrogram(dfn, figsize=(6, 3)); plt.show()
except ImportError:
    print("missingno no está instalado (opcional): pip install missingno. Los gráficos de arriba hacen lo mismo con matplotlib.")

## Paso 3 · El debate: ¿se pueden imputar todas las variables?
**No.** Lo que dijo el profesor al ver el reporte:

1. Con **60 %** de nulos, **paro** y reviso el **script SQL** de extracción (o le pido al ingeniero de datos que lo revise).
2. Si el script está bien y sigue igual, es un **problema de ingreso de datos**: se **reporta** y la variable **sale del análisis**.
   *"No puedes reconstruir la mitad de un mapa con la otra mitad."*
3. Si te obligan a usarla, las salidas son malas:
   - trabajar solo con el 40 % de filas completas → sesgas **todas** las variables (tiras información de las demás columnas);
   - rellenar con 0 → en una categórica creas una **categoría nueva**; en una numérica **destruyes la distribución**;
   - (*bootstrap* para muestrear el 40 %: posible, pero sigue siendo riesgoso.)

Mira qué pasa si te quedas solo con las filas completas:

In [ ]:
completas = dfn.dropna()
print(f"Filas originales: {len(dfn)}  ->  filas sin ningún NaN: {len(completas)}")
print("Tasa de abandono original:    ", round(df["abandono"].mean() * 100, 2), "%")
print("Tasa de abandono en completas:", round(completas["abandono"].mean() * 100, 2), "%")

Perdemos más del 60 % de las filas **y** la tasa de abandono cambia (porque `antiguedad_anios` falta más en quienes
abandonaron): justo el sesgo del que advierte el profesor.

### ✏️ Tu turno 1
Lista las columnas con **más de 5 %** de nulos y, para cada una, escribe en un comentario qué harías
(¿imputar o reportar y descartar?). Pista: `dfn.isna().mean() * 100`.

In [ ]:
# Tu código aquí

## Paso 4 · ¿Qué tipo de faltante es? Correlación de nulidad
El heatmap de missingno toma **solo las columnas con nulos**, convierte cada una en *¿falta? (True/False)* y calcula la
correlación entre esas columnas:

- **≈ 1** → cuando falta una, falta la otra (en la clase: `Marital_Status` y `Dependent_count`). Depende de otra
  característica → **MAR**. No conviene imputar con un método paramétrico (media/mediana); mejor uno **no paramétrico**.
- **≈ 0 con todas** → **potencial MCAR** ("potencial": hay que confirmarlo).
- **MNAR** → el más difícil: no muestra correlación con nada de lo que vemos.

In [ ]:
con_nulos = dfn.columns[dfn.isna().any()]
corr_nul = dfn[con_nulos].isna().corr().round(2)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr_nul, cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(len(con_nulos))); ax.set_xticklabels(con_nulos, rotation=60, ha="right")
ax.set_yticks(range(len(con_nulos))); ax.set_yticklabels(con_nulos)
for i in range(len(con_nulos)):
    for j in range(len(con_nulos)):
        ax.text(j, i, corr_nul.iloc[i, j], ha="center", va="center", fontsize=8)
fig.colorbar(im); ax.set_title("Correlación de nulidad"); plt.tight_layout(); plt.show()
corr_nul

Lectura (igual que el profe):
- `edad` ↔ `num_productos` = **1.0** → los vaciamos juntos → patrón **MAR**.
- `ingreso_mensual`, `reclamos_ultimo_anio` ≈ 0 con todas → **potenciales MCAR**.
- `antiguedad_anios` también sale ≈ 0… ¡pero la fabricamos dependiendo de `abandono`!

📌 **Extra (límite del heatmap):** solo compara columnas **que tienen nulos** entre sí. `abandono` no tiene nulos, así que
no aparece. Para eso sirve lo que hizo el profe al final: **ordenar** los datos por una columna y volver a mirar la matriz.

In [ ]:
ordenado = dfn.sort_values("abandono")
fig, ax = plt.subplots(figsize=(9, 4))
ax.imshow(ordenado.isna().to_numpy(), aspect="auto", cmap="gray_r", interpolation="nearest")
ax.set_xticks(range(dfn.shape[1])); ax.set_xticklabels(dfn.columns, rotation=60, ha="right")
ax.set_title("Matriz de nulidad ordenada por abandono (abajo = abandonó)")
plt.tight_layout(); plt.show()

dfn.groupby("abandono")["antiguedad_anios"].apply(lambda s: round(s.isna().mean() * 100, 1))

Abajo (clientes que abandonaron) la columna `antiguedad_anios` está llena de huecos: **no** es MCAR, depende de
`abandono`. La tabla lo confirma (es la `tasa_faltantes_por_grupo` de la clase 02).

### Dendrograma (agrupa variables por patrón de nulidad)
El profe lo mostró al final: junta en un mismo grupo las variables que "fallan parecido". Es **clustering jerárquico**
(aprendizaje no supervisado), lo verás en Machine Learning. Usa `scipy` (viene con scikit-learn):

In [ ]:
try:
    from scipy.cluster.hierarchy import dendrogram, linkage
    matriz = dfn.isna().T.astype(int).to_numpy()            # una fila por variable
    enlaces = linkage(matriz, method="average", metric="hamming")
    plt.figure(figsize=(8, 3))
    dendrogram(enlaces, labels=list(dfn.columns), leaf_rotation=60)
    plt.title("Dendrograma de nulidad"); plt.tight_layout(); plt.show()
except ImportError:
    print("scipy no está disponible; este gráfico es opcional.")

Las columnas sin nulos quedan en un grupo a altura 0; `edad` y `num_productos` se unen a altura 0 (patrón idéntico).

➡️ **Ejercicios 4 y 5** (`correlacion_nulidad`, `diagnostico_nulidad`): calculas esta matriz con sus casos borde y la
lees automáticamente ("posible MAR" / "potencial MCAR").

## Paso 5 · Imputar
Ya decidimos que `edad` y `num_productos` (60 %) **se descartan**. Imputamos las demás.

In [ ]:
base = dfn.drop(columns=["edad", "num_productos"])

### 5.1 Forward / backward fill (solo para series de tiempo)
`ffill` copia el valor de **arriba**, `bfill` el de **abajo**. Funciona cuando hay **continuidad** (series de tiempo);
aquí cada fila es un cliente distinto, así que el profe **no lo recomienda**.

In [ ]:
s = pd.Series([4, np.nan, 6, np.nan, np.nan, 9])
pd.DataFrame({"original": s, "ffill": s.ffill(), "bfill": s.bfill()})

### 5.2 Métodos paramétricos: media, mediana, cuartiles
La **media** es sensible a outliers (`ingreso_mensual` tiene cola larga), así que el profe la descarta. La **mediana**
no es sensible a outliers… pero **igual genera un pico**: cientos de filas con exactamente el mismo valor.
Mientras más faltantes, peor (referencia del profe: hasta ~30 % es "aceptable").

In [ ]:
mediana = base["ingreso_mensual"].median()
con_mediana = base["ingreso_mensual"].fillna(mediana)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharex=True)
base["ingreso_mensual"].plot.hist(bins=60, ax=axes[0], title="Original (sin los NaN)")
con_mediana.plot.hist(bins=60, ax=axes[1], color="tab:orange", title="Imputado con la mediana")
plt.tight_layout(); plt.show()
print("Valor más repetido tras la mediana:", con_mediana.value_counts().head(1).to_dict())

Ese pico es una "tendencia nueva" que la variable no tenía: estás **alterando** su comportamiento natural (y complica
transformaciones posteriores, como la logarítmica).

### 5.3 KNNImputer (avanzado, no paramétrico)
Idea de kNN con el dibujo del profe: llega una **X** y quiero saber si es triángulo o círculo → pregunto a sus **k vecinos
más cercanos**. Para imputar: busco las k filas más parecidas y relleno con lo que ellas tienen.
`weights="uniform"` = los 10 vecinos pesan lo mismo.

📌 Precisión: en `KNNImputer` el relleno es el **promedio** de los vecinos (en clasificación kNN se cuenta el más repetido).

In [ ]:
from sklearn.impute import KNNImputer

cols = ["ingreso_mensual", "antiguedad_anios", "reclamos_ultimo_anio", "abandono", "es_activo"]
imputer = KNNImputer(n_neighbors=10, weights="uniform")
valores = imputer.fit_transform(base[cols])            # devuelve un ARRAY de numpy
con_knn = pd.DataFrame(valores, columns=cols, index=base.index)   # lo volvemos DataFrame (¡con el índice!)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharex=True)
con_mediana.plot.hist(bins=60, ax=axes[0], color="tab:orange", title="Mediana")
con_knn["ingreso_mensual"].plot.hist(bins=60, ax=axes[1], color="tab:green", title="KNNImputer (k=10)")
plt.tight_layout(); plt.show()
print("Valor más repetido con KNN:", con_knn["ingreso_mensual"].value_counts().head(1).to_dict())

Distribución más suave: ya no hay un pico gigante. ➡️ **Ejercicio 6** (`imputar_knn`).

### ✏️ Tu turno 2
Repite la imputación KNN con `n_neighbors=2` y con `n_neighbors=50`. ¿Cuántas veces se repite el valor más frecuente
de `ingreso_mensual` en cada caso? ¿Por qué crees que con más vecinos los rellenos se parecen más entre sí?

In [ ]:
# Tu código aquí

### 5.4 IterativeImputer (avanzado: imputación como regresión)
Convierte la imputación en un problema de **regresión multivariada**: toma una columna con huecos como *salida*, las
demás como *entradas*, entrena con las filas que sí tienen dato y predice las que faltan; repite con cada columna y
varias rondas (`max_iter`). Más **robusto** pero más **costoso**. Es experimental en scikit-learn: hay que habilitarlo.
(El profe usó `ExtraTreesRegressor`; aquí con pocos árboles para que sea rápido.)

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (obligatorio antes del import)
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor

iterativo = IterativeImputer(
    estimator=ExtraTreesRegressor(n_estimators=10, random_state=123),
    max_iter=5, random_state=123,
)
con_iter = pd.DataFrame(iterativo.fit_transform(base[cols]), columns=cols, index=base.index)

resumen = pd.DataFrame({
    "original": base["ingreso_mensual"].describe(),
    "mediana": con_mediana.describe(),
    "knn": con_knn["ingreso_mensual"].describe(),
    "iterativo": con_iter["ingreso_mensual"].describe(),
}).round(1)
resumen

¿Cuál es mejor? El profe: **se decide al final, por el desempeño del modelo** (comparando KNN vs Iterative, o distintos
regresores, en tu protocolo experimental). Y para **categóricas**: primero un **encoding** (clase 09) y luego estos métodos.

### ✏️ Tu turno 3
Compara la desviación estándar (`.std()`) de `antiguedad_anios` original, imputada con la mediana y con KNN.
¿Cuál "aplana" más la variable?

In [ ]:
# Tu código aquí

## Paso 6 · Duplicados (rapidito, como el profe)
Copia, pregunta cuáles están duplicados, conserva el **primero** y elimina el resto.

In [ ]:
con_dup = pd.concat([df.head(5), df.head(2)])          # 7 filas, 2 repetidas
print("Duplicadas:", con_dup.duplicated().sum())
sin_dup = con_dup.drop_duplicates(keep="first")
print("Filas antes:", len(con_dup), "-> después:", len(sin_dup))

## Resumen
1. Reporte de nulos → ¿cuánto falta en cada variable?
2. Mucho faltante (60 %) → revisar script / reportar / **descartar**, no imputar.
3. Correlación de nulidad → ≈1 = MAR, ≈0 = potencial MCAR (confirmar ordenando/mezclando y por grupos).
4. Imputar: la mediana deja un **pico**; KNN e Iterative son más suaves; se elige por el desempeño del modelo.

Ahora: implementa `ejercicio.py` (orden 1 → 6) y evalúa.